# Exploratory analysis — TaskFlow feedback corpus

Quick look at the synthetic `sample_reviews.csv` used by the live pipeline:
volume over time, source mix, seed-topic distribution, and a dry-run of
sentiment + prioritization on a fixed window.

In [ ]:
import os
import sys
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt

ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if not os.path.exists(os.path.join(ROOT, "data", "sample_reviews.csv")):
    ROOT = os.getcwd()
sys.path.insert(0, ROOT)

df = pd.read_csv(os.path.join(ROOT, "data", "sample_reviews.csv"))
df["timestamp"] = pd.to_datetime(df["timestamp"])
df.head()

In [ ]:
print(f"Rows: {len(df)}")
print("Sources:\n", df["source"].value_counts())
print("\nSeed topics:\n", df["true_topic"].value_counts())
print("\nRating distribution:\n", df["rating"].value_counts().sort_index())

In [ ]:
daily = df.set_index("timestamp").resample("D").size()
ax = daily.plot(figsize=(10, 3), title="Feedback volume per day")
ax.set_ylabel("messages")
plt.tight_layout()
plt.show()

In [ ]:
from src.pipeline import FeedbackPipeline
from src.sentiment import SentimentAnalyzer
from src.prioritize import split_issues_and_requests

analyzer = SentimentAnalyzer(prefer_transformer=False)
pipe = FeedbackPipeline(analyzer=analyzer, min_messages=15, retrain_every=20)

items = [
    {
        "id": i,
        "text": row.text,
        "source": row.source,
        "timestamp": row.timestamp.strftime("%Y-%m-%d %H:%M:%S"),
        "rating": row.rating,
    }
    for i, row in enumerate(df.head(200).itertuples())
]
pipe.ingest(items, force_retrain=True)

print("Sentiment backend:", pipe.sentiment_backend)
print("Topic backend:", pipe.model_backend)
print("Topics discovered:", len([t for t in pipe.topic_info if t != -1]))

issues, requests = split_issues_and_requests(pipe.priorities)
print("\nTop issues:")
for i in issues[:5]:
    print(f"  [{i['priority_score']:5.1f}] {i['label']} (n={i['count']})")
print("\nTop feature requests:")
for r in requests[:5]:
    print(f"  [{r['count']} mentions] {r['label']}")

## Takeaways for the live dashboard

- The synthetic corpus intentionally spikes **crashes** in the most recent day
  so the priority engine has a clear "emerging issue" to surface.
- Feature-request templates use explicit wish/please-add phrasing so the
  request detector can separate them from complaints.
- Swap in a real CSV with columns `timestamp, text, source` and the same
  pipeline code applies unchanged.